In [ ]:
# In[1]:

import pandas as pd

# Load the metric_container.csv file for March 23, 2021
file_path = 'dataset/Bank/telemetry/2021_03_23/metric/metric_container.csv'
metric_container_df = pd.read_csv(file_path)

# Extract unique KPI names and their corresponding components (cmdb_id)
kpi_component_mapping = metric_container_df[['cmdb_id', 'kpi_name']].drop_duplicates()

kpi_component_mapping

```
Out[1]:
```
The `metric_container.csv` file for March 23, 2021, contains 1,741 unique combinations of KPI names and their corresponding components. These include KPIs related to CPU utilization, disk read/write operations, and container-specific metrics, among others. Components such as `Tomcat04`, `IG02`, and `dockerA1` are associated with these KPIs. For a complete understanding, further inspection of the full dataset is recommended.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name
0       Tomcat04                         OSLinux-CPU_CPU_CPUCpuUtil
1           IG02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTps
2           IG02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead
3           IG02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKWTps
4           IG02   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKWrite
...          ...                                                ...
595778  dockerA1  Container-DOCKER_CONTAINER_f861998c398e--bcou-...
595779  dockerA1  Container-DOCKER_CONTAINER_f861998c398e--bcou-...
595871  dockerA2  Container-DOCKER_CONTAINER_3bf443a64876--bcou-...
597017  dockerA2  Container-DOCKER_CONTAINER_76d31070b844--bcou-...
597018  dockerA2  Container-DOCKER_CONTAINER_76d31070b844--bcou-...

[1741 rows x 2 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

# Filter the dataset for the possible root cause components
root_cause_components = [
    'apache01', 'apache02', 'Tomcat01', 'Tomcat02', 'Tomcat04', 'Tomcat03',
    'MG01', 'MG02', 'IG01', 'IG02', 'Mysql01', 'Mysql02', 'Redis01', 'Redis02'
]
filtered_df = metric_container_df[metric_container_df['cmdb_id'].isin(root_cause_components)]

# Group by 'component-KPI' and calculate the global P95 threshold for each time series
filtered_df['component-KPI'] = filtered_df['cmdb_id'] + '-' + filtered_df['kpi_name']
p95_thresholds = filtered_df.groupby('component-KPI')['value'].quantile(0.95).reset_index()
p95_thresholds.columns = ['component-KPI', 'P95_threshold']

p95_thresholds

```
Out[2]:
```
The global P95 thresholds have been calculated for 1,723 unique 'component-KPI' time series from the possible root cause components on March 23, 2021. Each threshold represents the 95th percentile value of the respective KPI for a specific component. For example, `IG01-JVM-Memory_7778_JVM_Memory_HeapMemoryMax` has a P95 threshold of approximately 2.04 billion, while `apache02-OSLinux-OSLinux_ZABBIX_Host_Uptime` has a P95 threshold of approximately 7.66 million. Further analysis can be performed using these thresholds to identify anomalies or issues.

The original code execution output of IPython Kernel is also provided below for reference:

component-KPI  P95_threshold
0         IG01-JVM-Memory_7778_JVM_Memory_HeapMemoryMax   2.040136e+09
1       IG01-JVM-Memory_7778_JVM_Memory_HeapMemoryUsage   4.168419e+01
2        IG01-JVM-Memory_7778_JVM_Memory_HeapMemoryUsed   8.483727e+08
3      IG01-JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed   5.869456e+07
4        IG01-JVM-Operating System_7778_JVM_JVM_CPULoad   5.835350e-01
...                                                 ...            ...
1718              apache02-OSLinux-OSLinux_SWAP_SWAP_Si   0.000000e+00
1719              apache02-OSLinux-OSLinux_SWAP_SWAP_So   0.000000e+00
1720  apache02-OSLinux-OSLinux_SYSTEM_SYSTEM_Check-D...   1.000000e+00
1721  apache02-OSLinux-OSLinux_SYSTEM_SYSTEM_Check-H...   1.000000e+00
1722        apache02-OSLinux-OSLinux_ZABBIX_Host_Uptime   7.664431e+06

[1723 rows x 2 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pytz
from datetime import datetime

# Convert timestamps to UTC+8 and filter the data within the specified time range
utc8 = pytz.timezone('Asia/Shanghai')
filtered_df['timestamp_utc8'] = pd.to_datetime(filtered_df['timestamp'], unit='s').dt.tz_localize('UTC').dt.tz_convert(utc8)

start_time = utc8.localize(datetime(2021, 3, 23, 14, 30))
end_time = utc8.localize(datetime(2021, 3, 23, 15, 0))

time_filtered_df = filtered_df[(filtered_df['timestamp_utc8'] >= start_time) & (filtered_df['timestamp_utc8'] < end_time)]

time_filtered_df

```
Out[3]:
```
The data has been successfully filtered for the time range of March 23, 2021, from 14:30 to 15:00 (UTC+8). A total of 20,548 rows of telemetry data are available for the specified 'component-KPI' time series during this period. This filtered dataset can now be used for further analysis or anomaly detection.

The original code execution output of IPython Kernel is also provided below for reference:

timestamp   cmdb_id                                           kpi_name       value                                      component-KPI            timestamp_utc8
745141  1616481000  Tomcat01  OSLinux-OSLinux_FILESYSTEM_-tomcat_FSAvailable...  13615.0000  Tomcat01-OSLinux-OSLinux_FILESYSTEM_-tomcat_FS... 2021-03-23 14:30:00+08:00
745142  1616481000  Tomcat01      OSLinux-OSLinux_FILESYSTEM_-tomcat_FSCapacity     71.3752  Tomcat01-OSLinux-OSLinux_FILESYSTEM_-tomcat_FS... 2021-03-23 14:30:00+08:00
745143  1616481000  Tomcat01  OSLinux-OSLinux_FILESYSTEM_-tomcat_FSInodeUsed...      0.0471  Tomcat01-OSLinux-OSLinux_FILESYSTEM_-tomcat_FS... 2021-03-23 14:30:00+08:00
745144  1616481000  Tomcat01        OSLinux-OSLinux_FILESYSTEM_-tmp_FSUsedSpace     45.0000  Tomcat01-OSLinux-OSLinux_FILESYSTEM_-tmp_FSUse... 2021-03-23 14:30:00+08:00
745145  1616481000  Tomcat01  OSLinux-OSLinux_FILESYSTEM_-tmp_FSInodeUsedPer...      0.0153  Tomcat01-OSLinux-OSLinux_FILESYSTEM_-tmp_FSIno... 2021-03-23 14:30:00+08:00
...            ...       ...                                                ...         ...                                                ...                       ...
765684  1616482740   Mysql01                     Mysql-MySQL_3306_Opened Tables      0.0000             Mysql01-Mysql-MySQL_3306_Opened Tables 2021-03-23 14:59:00+08:00
765685  1616482740   Mysql01        Mysql-MySQL_3306_Innodb dblwr pages written      6.6167  Mysql01-Mysql-MySQL_3306_Innodb dblwr pages wr... 2021-03-23 14:59:00+08:00
765686  1616482740      MG01      OSLinux-OSLinux_FILESYSTEM_-tomcat_FSCapacity     80.4461  MG01-OSLinux-OSLinux_FILESYSTEM_-tomcat_FSCapa... 2021-03-23 14:59:00+08:00
765687  1616482740      MG01  OSLinux-OSLinux_FILESYSTEM_-tomcat_FSAvailable...   9300.0000  MG01-OSLinux-OSLinux_FILESYSTEM_-tomcat_FSAvai... 2021-03-23 14:59:00+08:00
765688  1616482740   Mysql01               Mysql-MySQL_3306_Innodb dblwr writes      0.5833       Mysql01-Mysql-MySQL_3306_Innodb dblwr writes 2021-03-23 14:59:00+08:00

[20548 rows x 6 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

# Merge the filtered dataset with the P95 thresholds to perform anomaly detection
anomaly_df = time_filtered_df.merge(p95_thresholds, on='component-KPI', how='left')

# Identify anomalies where the 'value' exceeds the corresponding P95 threshold
anomalies = anomaly_df[anomaly_df['value'] > anomaly_df['P95_threshold']]

# Select relevant columns for the anomaly report
anomaly_report = anomalies[['timestamp_utc8', 'cmdb_id', 'kpi_name', 'value', 'P95_threshold']]

anomaly_report

```
Out[4]:
```
Anomaly detection identified 380 data points in the filtered dataset where the 'value' exceeded the corresponding global P95 threshold for the respective 'component-KPI'. These anomalies span various components and KPIs, such as `Tomcat01` with high disk write throughput (`DSKWTps`) and `Mysql01` with locked transaction rows (`Max trx rows locked`). Each anomaly includes the timestamp, component, KPI, value, and threshold, providing detailed insights for further investigation.

The original code execution output of IPython Kernel is also provided below for reference:

timestamp_utc8   cmdb_id                                           kpi_name         value  P95_threshold
64    2021-03-23 14:30:00+08:00  Tomcat01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPer...  4.000000e-01   3.000000e-01
67    2021-03-23 14:30:00+08:00  Tomcat01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKWTps  1.200000e+00   1.183333e+00
76    2021-03-23 14:30:00+08:00  Tomcat01    OSLinux-OSLinux_NETWORK_ens160_NETBandwidthUtil  5.600000e-03   5.300000e-03
132   2021-03-23 14:30:00+08:00  Tomcat01    OSLinux-OSLinux_NETWORK_ens160_NETKBTotalPerSec  5.816390e+05   5.812586e+05
173   2021-03-23 14:30:00+08:00  Tomcat04                         OSLinux-CPU_CPU_CPUSysTime  1.123100e+00   1.091078e+00
...                         ...       ...                                                ...           ...            ...
20385 2021-03-23 14:59:00+08:00      MG01          JVM-Memory_7779_JVM_Memory_HeapMemoryUsed  1.011772e+09   9.953184e+08
20411 2021-03-23 14:59:00+08:00      MG01                         OSLinux-CPU_CPU_CPUCpuUtil  2.973160e+01   2.883720e+01
20486 2021-03-23 14:59:00+08:00   Mysql01               Mysql-MySQL_3306_Max trx rows locked  1.000000e+00   0.000000e+00
20502 2021-03-23 14:59:00+08:00   Mysql01                Mysql-MySQL_3306_MaxTrxRowsModified  1.000000e+00   0.000000e+00
20531 2021-03-23 14:59:00+08:00   Mysql01      Mysql-MySQL_3306_Innodb os log pending fsyncs  1.000000e+00   0.000000e+00

[380 rows x 5 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

# Group consecutive anomalies for each 'component-KPI'
anomaly_df['is_consecutive'] = (anomaly_df['component-KPI'] != anomaly_df['component-KPI'].shift()) | \
                               (anomaly_df['timestamp_utc8'] - anomaly_df['timestamp_utc8'].shift() > pd.Timedelta(minutes=1))

anomaly_df['fault_group'] = anomaly_df['is_consecutive'].cumsum()

# Filter out isolated noise spikes where the threshold breach is <= 50% of the extremal value
anomaly_df['threshold_breach'] = anomaly_df['value'] - anomaly_df['P95_threshold']
faults = anomaly_df.groupby('fault_group').filter(lambda group: group['threshold_breach'].max() > 0.5 * group['threshold_breach'].max())

# Extract fault details: start and end timestamps, components, KPIs, and values
fault_details = faults.groupby('fault_group').agg(
    start_timestamp=('timestamp_utc8', 'min'),
    end_timestamp=('timestamp_utc8', 'max'),
    component=('cmdb_id', 'first'),
    kpi=('kpi_name', 'first'),
    max_value=('value', 'max')
).reset_index(drop=True)

fault_details

```
Out[5]:
```
Fault detection identified 380 grouped anomalies across various components and KPIs in the filtered dataset. Each fault includes details such as the start and end timestamps, component, KPI, and the maximum value observed during the fault. For example, `Tomcat01` experienced a fault in `OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKWTps` at 14:30 with a maximum value of 1.2, and `Mysql01` had a fault in `Mysql-MySQL_3306_Max trx rows locked` at 14:59 with a maximum value of 1. These faults provide critical insights for diagnosing system issues.

The original code execution output of IPython Kernel is also provided below for reference:

start_timestamp             end_timestamp component                                                kpi     max_value
0   2021-03-23 14:30:00+08:00 2021-03-23 14:30:00+08:00  Tomcat01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPer...  4.000000e-01
1   2021-03-23 14:30:00+08:00 2021-03-23 14:30:00+08:00  Tomcat01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKWTps  1.200000e+00
2   2021-03-23 14:30:00+08:00 2021-03-23 14:30:00+08:00  Tomcat01    OSLinux-OSLinux_NETWORK_ens160_NETBandwidthUtil  5.600000e-03
3   2021-03-23 14:30:00+08:00 2021-03-23 14:30:00+08:00  Tomcat01    OSLinux-OSLinux_NETWORK_ens160_NETKBTotalPerSec  5.816390e+05
4   2021-03-23 14:30:00+08:00 2021-03-23 14:30:00+08:00  Tomcat04                         OSLinux-CPU_CPU_CPUSysTime  1.123100e+00
..                        ...                       ...       ...                                                ...           ...
375 2021-03-23 14:59:00+08:00 2021-03-23 14:59:00+08:00      MG01          JVM-Memory_7779_JVM_Memory_HeapMemoryUsed  1.011772e+09
376 2021-03-23 14:59:00+08:00 2021-03-23 14:59:00+08:00      MG01                         OSLinux-CPU_CPU_CPUCpuUtil  2.973160e+01
377 2021-03-23 14:59:00+08:00 2021-03-23 14:59:00+08:00   Mysql01               Mysql-MySQL_3306_Max trx rows locked  1.000000e+00
378 2021-03-23 14:59:00+08:00 2021-03-23 14:59:00+08:00   Mysql01                Mysql-MySQL_3306_MaxTrxRowsModified  1.000000e+00
379 2021-03-23 14:59:00+08:00 2021-03-23 14:59:00+08:00   Mysql01      Mysql-MySQL_3306_Innodb os log pending fsyncs  1.000000e+00

[380 rows x 5 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

The Executor failed to complete the instruction, please re-write a new instruction for Executor.

```
Out[6]:
```
The Executor failed to complete the instruction, please re-write a new instruction for Executor.```
```

In [ ]:
# In[7]:

# Add the 'component-KPI' column to fault_details for merging
fault_details['component-KPI'] = fault_details['component'] + '-' + fault_details['kpi']

# Calculate the deviation for each fault
fault_details = fault_details.merge(p95_thresholds, on='component-KPI', how='left')
fault_details['deviation'] = fault_details['max_value'] - fault_details['P95_threshold']

# Identify the fault with the largest deviation
largest_deviation_fault = fault_details.loc[fault_details['deviation'].idxmax()]

# Determine the level (service, container, or node) based on the component name
component = largest_deviation_fault['component']
if component.startswith('apache') or component.startswith('Tomcat'):
    level = 'service'
elif component.startswith('docker') or component.startswith('MG') or component.startswith('IG'):
    level = 'container'
else:
    level = 'node'

largest_deviation_fault, level

```
Out[7]:
```
The fault with the most significant deviation from the threshold occurred in the `Tomcat01` component at the service level. The KPI involved was `Tomcat-MEMORY_7441-MEMORY_JVMUsedMemory`, with a maximum value of 2,149,914,760, exceeding the P95 threshold of 2,110,085,498.8 by a deviation of 39,829,261.2.

The original code execution output of IPython Kernel is also provided below for reference:

(start_timestamp                           2021-03-23 14:36:00+08:00
end_timestamp                             2021-03-23 14:36:00+08:00
component                                                  Tomcat01
kpi                         Tomcat-MEMORY_7441-MEMORY_JVMUsedMemory
max_value                                              2149914760.0
component-KPI      Tomcat01-Tomcat-MEMORY_7441-MEMORY_JVMUsedMemory
P95_threshold                                          2110085498.8
deviation                                                39829261.2
Name: 79, dtype: object, 'service')```
```

In [ ]:
# In[8]:

# Load the trace_span.csv file for March 23, 2021
trace_file_path = 'dataset/Bank/telemetry/2021_03_23/trace/trace_span.csv'
trace_span_df = pd.read_csv(trace_file_path)

# Convert timestamps to UTC+8 and filter the data within the failure period (14:30 to 15:00 UTC+8)
trace_span_df['timestamp_utc8'] = pd.to_datetime(trace_span_df['timestamp'], unit='ms').dt.tz_localize('UTC').dt.tz_convert(utc8)
trace_filtered_df = trace_span_df[(trace_span_df['timestamp_utc8'] >= start_time) & (trace_span_df['timestamp_utc8'] < end_time)]

# Filter traces involving Tomcat01
tomcat01_traces = trace_filtered_df[trace_filtered_df['cmdb_id'] == 'Tomcat01']

# Check if Tomcat01 is the most downstream faulty service in the call chain
# Extract parent-child relationships for the traces involving Tomcat01
trace_details = tomcat01_traces[['trace_id', 'span_id', 'parent_id', 'cmdb_id']]

trace_details

```
Out[8]:
```
No traces involving `Tomcat01` were found in the `trace_span.csv` file for the failure period (14:30 to 15:00 UTC+8) on March 23, 2021. Therefore, it cannot be determined whether `Tomcat01` is the most downstream faulty service in the call chain during this time.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [trace_id, span_id, parent_id, cmdb_id]
Index: []```
```

In [ ]:
# In[9]:

# Load the log_service.csv file for March 23, 2021
log_file_path = 'dataset/Bank/telemetry/2021_03_23/log/log_service.csv'
log_service_df = pd.read_csv(log_file_path)

# Convert timestamps to UTC+8 and filter the data within the failure period (14:30 to 15:00 UTC+8)
log_service_df['timestamp_utc8'] = pd.to_datetime(log_service_df['timestamp'], unit='s').dt.tz_localize('UTC').dt.tz_convert(utc8)
tomcat01_logs = log_service_df[(log_service_df['cmdb_id'] == 'Tomcat01') & 
                                (log_service_df['timestamp_utc8'] >= start_time) & 
                                (log_service_df['timestamp_utc8'] < end_time)]

# Extract logs that may indicate resource issues, errors, or anomalies
resource_issue_keywords = ['error', 'fail', 'exception', 'memory', 'cpu', 'disk', 'gc', 'heap', 'latency']
filtered_logs = tomcat01_logs[tomcat01_logs['value'].str.contains('|'.join(resource_issue_keywords), case=False, na=False)]

filtered_logs[['timestamp_utc8', 'log_name', 'value']]

```
Out[9]:
```
During the failure period (14:30 to 15:00 UTC+8) on March 23, 2021, multiple logs related to `Tomcat01` were identified, all indicating "GC (Allocation Failure)" events. These logs suggest potential memory allocation issues, with timestamps spanning the entire failure period. Each log entry provides details about garbage collection events, which may point to resource constraints or anomalies in the JVM memory management of `Tomcat01`.

The original code execution output of IPython Kernel is also provided below for reference:

timestamp_utc8 log_name                                              value
2656678 2021-03-23 14:30:18+08:00       gc  706583.858: [GC (Allocation Failure) 2021-03-2...
2656679 2021-03-23 14:31:32+08:00       gc  706657.418: [GC (Allocation Failure) 2021-03-2...
2656680 2021-03-23 14:32:49+08:00       gc  706734.651: [GC (Allocation Failure) 2021-03-2...
2656681 2021-03-23 14:33:57+08:00       gc  706802.255: [GC (Allocation Failure) 2021-03-2...
2656682 2021-03-23 14:35:12+08:00       gc  706877.432: [GC (Allocation Failure) 2021-03-2...
2656683 2021-03-23 14:36:30+08:00       gc  706955.063: [GC (Allocation Failure) 2021-03-2...
2656684 2021-03-23 14:37:50+08:00       gc  707035.203: [GC (Allocation Failure) 2021-03-2...
2656685 2021-03-23 14:39:19+08:00       gc  707124.017: [GC (Allocation Failure) 2021-03-2...
2656686 2021-03-23 14:40:37+08:00       gc  707202.382: [GC (Allocation Failure) 2021-03-2...
2656687 2021-03-23 14:41:53+08:00       gc  707278.634: [GC (Allocation Failure) 2021-03-2...
2656688 2021-03-23 14:42:55+08:00       gc  707339.984: [GC (Allocation Failure) 2021-03-2...
2656689 2021-03-23 14:44:07+08:00       gc  707412.051: [GC (Allocation Failure) 2021-03-2...
2656690 2021-03-23 14:45:30+08:00       gc  707495.389: [GC (Allocation Failure) 2021-03-2...
2656691 2021-03-23 14:46:42+08:00       gc  707567.150: [GC (Allocation Failure) 2021-03-2...
2656692 2021-03-23 14:47:55+08:00       gc  707640.702: [GC (Allocation Failure) 2021-03-2...
2656693 2021-03-23 14:49:20+08:00       gc  707725.026: [GC (Allocation Failure) 2021-03-2...
2656694 2021-03-23 14:50:38+08:00       gc  707803.197: [GC (Allocation Failure) 2021-03-2...
2656695 2021-03-23 14:51:58+08:00       gc  707883.627: [GC (Allocation Failure) 2021-03-2...
2656696 2021-03-23 14:53:26+08:00       gc  707971.474: [GC (Allocation Failure) 2021-03-2...
2656697 2021-03-23 14:54:47+08:00       gc  708052.349: [GC (Allocation Failure) 2021-03-2...
2656698 2021-03-23 14:55:57+08:00       gc  708122.807: [GC (Allocation Failure) 2021-03-2...
2656699 2021-03-23 14:57:16+08:00       gc  708201.096: [GC (Allocation Failure) 2021-03-2...
2656700 2021-03-23 14:58:27+08:00       gc  708271.937: [GC (Allocation Failure) 2021-03-2...
2656701 2021-03-23 14:59:36+08:00       gc  708341.384: [GC (Allocation Failure) 2021-03-2...```
```